# 03 — Feature engineering and preprocessing design

## Purpose

This notebook turns the prepared PD cohort into reproducible model inputs while keeping the final test period separate. It defines two feature policies: a borrower/application/credit-history baseline and a comparison that additionally includes Lending Club grade and pricing information.

## Input and chronological split

Input: `data/processed/pd_cohort_2011_2015_36m.parquet`.

- Training: loans issued in 2011–2013.
- Validation: loans issued in 2014.
- Held-out test: loans issued in 2015.

The test period is reserved for final evaluation. Preprocessing is fitted using training records only and applied without refitting to validation records.

## Feature policies

The baseline has **21 input columns**: 15 numerical and 6 categorical. It excludes IDs, loan status, target, issue date as a predictor, repayment outcomes, and Lending Club's grade/pricing fields. The comparison adds `grade`, `sub_grade`, `int_rate`, and `installment`, for **25 input columns**. It should be interpreted as an already-underwritten/priced-loan comparison, not an independent pre-approval score.

`verification_status` is retained provisionally in the baseline. Its availability at the exact proposed decision point should be checked; a stricter sensitivity analysis can exclude it.

In [4]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve().parent
pd_file = (
    project_root
    / "data"
    / "processed"
    / "pd_cohort_2011_2015_36m.parquet"
)

data = pd.read_parquet(pd_file)

assert data["id"].is_unique
assert data["issue_month"].notna().all()
assert set(data["charged_off"].unique()) == {0, 1}

train = data.loc[
    data["issue_month"].between("2011-01-01", "2013-12-01")
].copy()

validation = data.loc[
    data["issue_month"].between("2014-01-01", "2014-12-01")
].copy()

test = data.loc[
    data["issue_month"].between("2015-01-01", "2015-12-01")
].copy()

assert len(train) + len(validation) + len(test) == len(data)
assert train["id"].is_unique
assert validation["id"].is_unique
assert test["id"].is_unique
assert train["issue_month"].max() < validation["issue_month"].min()
assert validation["issue_month"].max() < test["issue_month"].min()

for name, subset in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    events = int(subset["charged_off"].sum())
    print(
        f"{name}: {len(subset):,} loans | "
        f"{events:,} charged off | "
        f"{events / len(subset):.2%} observed charge-off rate | "
        f"{subset['issue_month'].min().date()} to "
        f"{subset['issue_month'].max().date()}"
    )

train: 157,993 loans | 19,780 charged off | 12.52% observed charge-off rate | 2011-01-01 to 2013-12-01
validation: 162,570 loans | 22,315 charged off | 13.73% observed charge-off rate | 2014-01-01 to 2014-12-01
test: 283,026 loans | 42,132 charged off | 14.89% observed charge-off rate | 2015-01-01 to 2015-12-01


define feature sets

In [5]:
baseline_features = [
    "loan_amnt",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "earliest_cr_line",
    "fico_range_low",
    "fico_range_high",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "application_type",
    "addr_state",
    "collections_12_mths_ex_med",
    "acc_now_delinq",
    "pub_rec_bankruptcies",
]

lender_pricing_features = [
    "grade",
    "sub_grade",
    "int_rate",
    "installment",
]

excluded_from_pd = {
    "id",
    "issue_month",
    "loan_status",
    "charged_off",
    "funded_amnt",
    "term",
    "initial_list_status",
}

assert len(baseline_features) == len(set(baseline_features))
assert len(lender_pricing_features) == len(set(lender_pricing_features))
assert not set(baseline_features) & set(lender_pricing_features)
assert not set(baseline_features + lender_pricing_features) & excluded_from_pd

missing = (
    set(baseline_features + lender_pricing_features)
    - set(data.columns)
)
assert not missing, f"Missing columns: {sorted(missing)}"

print("Baseline feature count:", len(baseline_features))
print("Lender-pricing add-on count:", len(lender_pricing_features))
print("Baseline features:", baseline_features)
print("Lender-pricing add-ons:", lender_pricing_features)

print("\nColumns deliberately outside both feature sets:")
print(
    sorted(
        set(data.columns)
        - set(baseline_features)
        - set(lender_pricing_features)
    )
)

Baseline feature count: 22
Lender-pricing add-on count: 4
Baseline features: ['loan_amnt', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'dti', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'application_type', 'addr_state', 'collections_12_mths_ex_med', 'acc_now_delinq', 'pub_rec_bankruptcies']
Lender-pricing add-ons: ['grade', 'sub_grade', 'int_rate', 'installment']

Columns deliberately outside both feature sets:
['charged_off', 'funded_amnt', 'id', 'initial_list_status', 'issue_month', 'loan_status', 'term']


audit credit-line dates

In [6]:
def credit_date_audit(frame, name):
    opened = pd.to_datetime(
        frame["earliest_cr_line"],
        format="%b-%Y",
        errors="coerce",
    )

    future = opened.gt(frame["issue_month"])
    age_months = (
        (frame["issue_month"].dt.year - opened.dt.year) * 12
        + (frame["issue_month"].dt.month - opened.dt.month)
    )

    print(f"\n{name}")
    print("Rows:", f"{len(frame):,}")
    print("Missing/unparseable dates:", f"{opened.isna().sum():,}")
    print("Credit-line date after issue month:", f"{future.sum():,}")
    print("Earliest credit-line date:", opened.min())
    print("Latest credit-line date:", opened.max())
    print("Age in months—min/median/max:",
          age_months.min(), age_months.median(), age_months.max())

    if future.any():
        print("Examples of future dates:")
        print(
            frame.loc[
                future, ["id", "issue_month", "earliest_cr_line"]
            ]
            .head(5)
            .to_string(index=False)
        )

for name, frame in [
    ("train", train),
    ("validation", validation),
    ("test", test),
]:
    credit_date_audit(frame, name)


train
Rows: 157,993
Missing/unparseable dates: 0
Credit-line date after issue month: 0
Earliest credit-line date: 1946-01-01 00:00:00
Latest credit-line date: 2010-11-01 00:00:00
Age in months—min/median/max: 36 165.0 785



validation
Rows: 162,570
Missing/unparseable dates: 0
Credit-line date after issue month: 0
Earliest credit-line date: 1948-01-01 00:00:00
Latest credit-line date: 2011-11-01 00:00:00
Age in months—min/median/max: 37 178.0 793

test
Rows: 283,026
Missing/unparseable dates: 0
Credit-line date after issue month: 0
Earliest credit-line date: 1944-01-01 00:00:00
Latest credit-line date: 2012-11-01 00:00:00
Age in months—min/median/max: 37 180.0 852


deterministic features

In [7]:
import numpy as np

def make_row_features(frame):
    result = frame.copy()

    opened = pd.to_datetime(
        result["earliest_cr_line"],
        format="%b-%Y",
        errors="coerce",
    )

    result["credit_history_months"] = (
        (result["issue_month"].dt.year - opened.dt.year) * 12
        + (result["issue_month"].dt.month - opened.dt.month)
    )

    # Keep missing employment length explicit rather than guessing years.
    result["emp_length"] = (
        result["emp_length"]
        .astype("string")
        .fillna("Missing")
    )

    # DTI of 999 was observed in the audit; treat it as suspect/missing,
    # without changing other high DTI values yet.
    result["dti_999_flag"] = result["dti"].eq(999).astype("int8")
    result["dti"] = result["dti"].mask(result["dti"].eq(999))

    # Nonnegative loan amounts/incomes were established in the audit.
    # log1p is defined for zero and reduces the influence of large values.
    result["annual_inc_log"] = np.log1p(result["annual_inc"])
    result["revol_bal_log"] = np.log1p(result["revol_bal"])

    return result.drop(columns=["earliest_cr_line"])

train_fe = make_row_features(train)
validation_fe = make_row_features(validation)
test_fe = make_row_features(test)

for name, frame in [
    ("train", train_fe),
    ("validation", validation_fe),
    ("test", test_fe),
]:
    print(f"\n{name}: {frame.shape}")
    print(
        "Credit-history months min/median/max:",
        frame["credit_history_months"].min(),
        frame["credit_history_months"].median(),
        frame["credit_history_months"].max(),
    )
    print("Missing employment labels:",
          int(frame["emp_length"].eq("Missing").sum()))
    print("DTI=999 flags:", int(frame["dti_999_flag"].sum()))
    print("Missing DTI after flagging:", int(frame["dti"].isna().sum()))
    print(
        "Non-finite log-income values:",
        int((~np.isfinite(frame["annual_inc_log"])).sum()),
    )


train: (157993, 36)
Credit-history months min/median/max: 36 165.0 785
Missing employment labels: 7279
DTI=999 flags: 0
Missing DTI after flagging: 0
Non-finite log-income values: 0

validation: (162570, 36)
Credit-history months min/median/max: 37 178.0 793
Missing employment labels: 9646
DTI=999 flags: 0
Missing DTI after flagging: 0
Non-finite log-income values: 0

test: (283026, 36)
Credit-history months min/median/max: 37 180.0 852
Missing employment labels: 18960
DTI=999 flags: 1
Missing DTI after flagging: 3
Non-finite log-income values: 0


define model inputs

In [8]:
baseline_model_features = [
    col for col in baseline_features
    if col != "earliest_cr_line"
] + [
    "credit_history_months",
    "dti_999_flag",
    "annual_inc_log",
    "revol_bal_log",
]

pricing_model_features = (
    baseline_model_features + lender_pricing_features
)

numeric_features = [
    "loan_amnt",
    "annual_inc_log",
    "dti",
    "fico_range_low",
    "fico_range_high",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal_log",
    "revol_util",
    "total_acc",
    "collections_12_mths_ex_med",
    "acc_now_delinq",
    "pub_rec_bankruptcies",
    "credit_history_months",
    "dti_999_flag",
]

categorical_features = [
    "emp_length",
    "home_ownership",
    "verification_status",
    "purpose",
    "application_type",
    "addr_state",
]

pricing_numeric_features = ["int_rate", "installment"]
pricing_categorical_features = ["grade", "sub_grade"]

# Avoid using raw income/balance alongside their log versions.
baseline_model_features = [
    col for col in baseline_model_features
    if col not in {"annual_inc", "revol_bal"}
]
pricing_model_features = (
    baseline_model_features + lender_pricing_features
)

assert set(baseline_model_features) == (
    set(numeric_features) | set(categorical_features)
)
assert not set(numeric_features) & set(categorical_features)
assert len(baseline_model_features) == len(set(baseline_model_features))
assert not set(baseline_model_features) & {
    "id", "loan_status", "charged_off", "issue_month",
    "total_rec_prncp", "recoveries",
}

for name, frame in [
    ("train", train_fe),
    ("validation", validation_fe),
    ("test", test_fe),
]:
    assert set(pricing_model_features).issubset(frame.columns)
    print(
        f"{name}: baseline={len(baseline_model_features)} columns; "
        f"with pricing={len(pricing_model_features)} columns"
    )

print("\nBaseline numeric:", numeric_features)
print("Baseline categorical:", categorical_features)
print("Pricing-only numeric:", pricing_numeric_features)
print("Pricing-only categorical:", pricing_categorical_features)

train: baseline=23 columns; with pricing=27 columns
validation: baseline=23 columns; with pricing=27 columns
test: baseline=23 columns; with pricing=27 columns

Baseline numeric: ['loan_amnt', 'annual_inc_log', 'dti', 'fico_range_low', 'fico_range_high', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal_log', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'acc_now_delinq', 'pub_rec_bankruptcies', 'credit_history_months', 'dti_999_flag']
Baseline categorical: ['emp_length', 'home_ownership', 'verification_status', 'purpose', 'application_type', 'addr_state']
Pricing-only numeric: ['int_rate', 'installment']
Pricing-only categorical: ['grade', 'sub_grade']


finalize baseline inputs

In [9]:
def add_fico_midpoint(frame):
    result = frame.copy()
    result["fico_midpoint"] = (
        result["fico_range_low"] + result["fico_range_high"]
    ) / 2
    return result

train_fe = add_fico_midpoint(train_fe)
validation_fe = add_fico_midpoint(validation_fe)
test_fe = add_fico_midpoint(test_fe)

numeric_features = [
    col for col in numeric_features
    if col not in {
        "fico_range_low",
        "fico_range_high",
        "dti_999_flag",
    }
] + ["fico_midpoint"]

baseline_model_features = numeric_features + categorical_features
pricing_model_features = (
    baseline_model_features
    + pricing_numeric_features
    + pricing_categorical_features
)

for name, frame in [
    ("train", train_fe),
    ("validation", validation_fe),
    ("test", test_fe),
]:
    assert set(pricing_model_features).issubset(frame.columns)
    assert frame["fico_midpoint"].notna().all()
    assert frame["fico_midpoint"].between(300, 850).all()
    print(
        f"{name}: baseline={len(baseline_model_features)}; "
        f"pricing comparison={len(pricing_model_features)}"
    )

print("\nFinal baseline numeric features:", numeric_features)
print("Final baseline categorical features:", categorical_features)

train: baseline=21; pricing comparison=25
validation: baseline=21; pricing comparison=25
test: baseline=21; pricing comparison=25

Final baseline numeric features: ['loan_amnt', 'annual_inc_log', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal_log', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'acc_now_delinq', 'pub_rec_bankruptcies', 'credit_history_months', 'fico_midpoint']
Final baseline categorical features: ['emp_length', 'home_ownership', 'verification_status', 'purpose', 'application_type', 'addr_state']


fit baseline preprocessor

In [10]:
import inspect

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

encoder_kwargs = {"handle_unknown": "ignore"}
if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
    encoder_kwargs["sparse_output"] = True
else:
    encoder_kwargs["sparse"] = True

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(**encoder_kwargs)),
])

baseline_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

X_train = train_fe[baseline_model_features]
X_validation = validation_fe[baseline_model_features]

y_train = train_fe["charged_off"]
y_validation = validation_fe["charged_off"]

X_train_prepared = baseline_preprocessor.fit_transform(X_train)
X_validation_prepared = baseline_preprocessor.transform(X_validation)

print("Train matrix:", X_train_prepared.shape)
print("Validation matrix:", X_validation_prepared.shape)
print(
    "Same number of transformed columns:",
    X_train_prepared.shape[1] == X_validation_prepared.shape[1],
)
print("Train labels:", y_train.value_counts().sort_index().to_dict())
print("Validation labels:", y_validation.value_counts().sort_index().to_dict())

train_values = (
    X_train_prepared.data
    if hasattr(X_train_prepared, "data") and
    not isinstance(X_train_prepared, np.ndarray)
    else np.asarray(X_train_prepared).ravel()
)
validation_values = (
    X_validation_prepared.data
    if hasattr(X_validation_prepared, "data") and
    not isinstance(X_validation_prepared, np.ndarray)
    else np.asarray(X_validation_prepared).ravel()
)

print("All training values finite:", np.isfinite(train_values).all())
print("All validation values finite:", np.isfinite(validation_values).all())

Train matrix: (157993, 98)
Validation matrix: (162570, 98)
Same number of transformed columns: True
Train labels: {0: 138213, 1: 19780}
Validation labels: {0: 140255, 1: 22315}
All training values finite: True
All validation values finite: True


inspect encoded features

In [11]:
feature_names = baseline_preprocessor.get_feature_names_out()
encoder = (
    baseline_preprocessor
    .named_transformers_["cat"]
    .named_steps["encoder"]
)

print("Transformed feature count:", len(feature_names))
print("First 25 transformed features:")
print(feature_names[:25].tolist())

print("\nEncoded categories and unseen validation values:")
for col, learned in zip(categorical_features, encoder.categories_):
    learned_set = set(learned.tolist())
    validation_values = set(
        X_validation[col].dropna().unique().tolist()
    )
    unseen = sorted(
        validation_values - learned_set,
        key=str,
    )
    print(
        f"{col}: {len(learned_set)} trained categories; "
        f"unseen in validation: {unseen}"
    )

print("\nOutput matrix formats:")
print("Train:", type(X_train_prepared).__name__)
print("Validation:", type(X_validation_prepared).__name__)

assert len(feature_names) == X_train_prepared.shape[1]
assert X_train_prepared.shape[1] == X_validation_prepared.shape[1]

Transformed feature count: 98
First 25 transformed features:
['num__loan_amnt', 'num__annual_inc_log', 'num__dti', 'num__delinq_2yrs', 'num__inq_last_6mths', 'num__open_acc', 'num__pub_rec', 'num__revol_bal_log', 'num__revol_util', 'num__total_acc', 'num__collections_12_mths_ex_med', 'num__acc_now_delinq', 'num__pub_rec_bankruptcies', 'num__credit_history_months', 'num__fico_midpoint', 'cat__emp_length_1 year', 'cat__emp_length_10+ years', 'cat__emp_length_2 years', 'cat__emp_length_3 years', 'cat__emp_length_4 years', 'cat__emp_length_5 years', 'cat__emp_length_6 years', 'cat__emp_length_7 years', 'cat__emp_length_8 years', 'cat__emp_length_9 years']

Encoded categories and unseen validation values:
emp_length: 12 trained categories; unseen in validation: []
home_ownership: 5 trained categories; unseen in validation: ['ANY']
verification_status: 3 trained categories; unseen in validation: []
purpose: 13 trained categories; unseen in validation: []
application_type: 1 trained categorie

count unseen rows

In [12]:
print("Unseen validation values by categorical feature")

for col, learned in zip(categorical_features, encoder.categories_):
    known = set(learned.tolist())
    unseen_mask = ~X_validation[col].isin(known)

    print(
        f"{col}: {int(unseen_mask.sum()):,} rows "
        f"({unseen_mask.mean():.3%})"
    )

    if unseen_mask.any():
        print(
            X_validation.loc[unseen_mask, col]
            .value_counts(dropna=False)
            .to_string()
        )

Unseen validation values by categorical feature
emp_length: 0 rows (0.000%)
home_ownership: 1 rows (0.001%)
home_ownership
ANY    1
verification_status: 0 rows (0.000%)
purpose: 0 rows (0.000%)
application_type: 0 rows (0.000%)
addr_state: 1 rows (0.001%)
addr_state
ME    1


pricing comparison

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np

pricing_num = numeric_features + pricing_numeric_features
pricing_cat = categorical_features + pricing_categorical_features

pricing_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            pricing_num,
        ),
        (
            "cat",
            Pipeline([
                ("encoder", OneHotEncoder(**encoder_kwargs)),
            ]),
            pricing_cat,
        ),
    ],
    remainder="drop",
)

X_train_pricing = train_fe[pricing_model_features]
X_validation_pricing = validation_fe[pricing_model_features]

X_train_pricing_prepared = pricing_preprocessor.fit_transform(
    X_train_pricing
)
X_validation_pricing_prepared = pricing_preprocessor.transform(
    X_validation_pricing
)

def matrix_is_finite(matrix):
    values = matrix.data if hasattr(matrix, "data") else np.asarray(matrix)
    return bool(np.isfinite(values).all())

print("Pricing train matrix:", X_train_pricing_prepared.shape)
print("Pricing validation matrix:", X_validation_pricing_prepared.shape)
print(
    "Same transformed columns:",
    X_train_pricing_prepared.shape[1]
    == X_validation_pricing_prepared.shape[1],
)
print("Train values finite:", matrix_is_finite(X_train_pricing_prepared))
print(
    "Validation values finite:",
    matrix_is_finite(X_validation_pricing_prepared)
)

pricing_encoder = (
    pricing_preprocessor
    .named_transformers_["cat"]
    .named_steps["encoder"]
)

for col, known_values in zip(pricing_cat, pricing_encoder.categories_):
    if col in pricing_categorical_features:
        unseen = ~X_validation_pricing[col].isin(known_values)
        print(
            f"{col}: {len(known_values)} training categories; "
            f"{int(unseen.sum()):,} unseen validation rows"
        )

Pricing train matrix: (157993, 142)
Pricing validation matrix: (162570, 142)
Same transformed columns: True
Train values finite: True
Validation values finite: True
grade: 7 training categories; 0 unseen validation rows
sub_grade: 35 training categories; 0 unseen validation rows


In [16]:
module_file = (
    Path.cwd().resolve().parent
    / "src"
    / "features"
    / "credit_features.py"
)

saved_code = module_file.read_text(encoding="utf-8")

print("File exists:", module_file.is_file())
print("Size (bytes):", module_file.stat().st_size)
print("Matches current notebook cell:", saved_code == module_code)

File exists: True
Size (bytes): 2816
Matches current notebook cell: True


In [17]:
project_root = Path.cwd().resolve().parent
features_dir = project_root / "src" / "features"
features_dir.mkdir(parents=True, exist_ok=True)

module_file = features_dir / "credit_features.py"

module_code = '''"""Reproducible feature rules for the Lending Club PD cohort."""

import numpy as np
import pandas as pd

NUMERIC_FEATURES = [
    "loan_amnt",
    "annual_inc_log",
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal_log",
    "revol_util",
    "total_acc",
    "collections_12_mths_ex_med",
    "acc_now_delinq",
    "pub_rec_bankruptcies",
    "credit_history_months",
    "fico_midpoint",
]

CATEGORICAL_FEATURES = [
    "emp_length",
    "home_ownership",
    "verification_status",
    "purpose",
    "application_type",
    "addr_state",
]

PRICING_NUMERIC_FEATURES = ["int_rate", "installment"]
PRICING_CATEGORICAL_FEATURES = ["grade", "sub_grade"]

BASELINE_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
PRICING_FEATURES = (
    BASELINE_FEATURES
    + PRICING_NUMERIC_FEATURES
    + PRICING_CATEGORICAL_FEATURES
)

SPLIT_PERIODS = {
    "train": ("2011-01-01", "2013-12-01"),
    "validation": ("2014-01-01", "2014-12-01"),
    "test": ("2015-01-01", "2015-12-01"),
}


def make_row_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Create deterministic features without fitting on any dataset."""
    result = frame.copy()

    opened = pd.to_datetime(
        result["earliest_cr_line"],
        format="%b-%Y",
        errors="coerce",
    )
    if opened.isna().any():
        raise ValueError("Unparseable earliest_cr_line values found.")
    if opened.gt(result["issue_month"]).any():
        raise ValueError("Credit-line dates after issue_month found.")

    result["credit_history_months"] = (
        (result["issue_month"].dt.year - opened.dt.year) * 12
        + (result["issue_month"].dt.month - opened.dt.month)
    )

    result["emp_length"] = (
        result["emp_length"].astype("string").fillna("Missing")
    )

    result["dti"] = result["dti"].mask(result["dti"].eq(999))
    result["annual_inc_log"] = np.log1p(result["annual_inc"])
    result["revol_bal_log"] = np.log1p(result["revol_bal"])
    result["fico_midpoint"] = (
        result["fico_range_low"] + result["fico_range_high"]
    ) / 2

    return result.drop(columns=["earliest_cr_line"])


def split_by_issue_month(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Return non-overlapping, chronological study periods."""
    if frame["issue_month"].isna().any():
        raise ValueError("Missing issue_month values found.")

    splits = {
        name: frame.loc[
            frame["issue_month"].between(start, end)
        ].copy()
        for name, (start, end) in SPLIT_PERIODS.items()
    }

    if sum(len(part) for part in splits.values()) != len(frame):
        raise ValueError("Some rows are outside the declared split periods.")

    return splits
'''

if module_file.exists():
    if module_file.read_text(encoding="utf-8") != module_code:
        raise RuntimeError(
            "credit_features.py differs from this notebook. "
            "Review the differences before replacing it."
        )
    print("Feature module already exists and matches; no rewrite needed.")
else:
    module_file.write_text(module_code, encoding="utf-8")
    print("Saved:", module_file)

Feature module already exists and matches; no rewrite needed.


fresh-state verification

In [18]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))

from src.features.credit_features import (
    BASELINE_FEATURES,
    PRICING_FEATURES,
    make_row_features,
    split_by_issue_month,
)

pd_file = (
    project_root
    / "data"
    / "processed"
    / "pd_cohort_2011_2015_36m.parquet"
)

data = pd.read_parquet(pd_file)
splits = split_by_issue_month(data)

expected = {
    "train": (157_993, 19_780),
    "validation": (162_570, 22_315),
    "test": (283_026, 42_132),
}

for name, frame in splits.items():
    featured = make_row_features(frame)

    assert (len(featured), int(featured["charged_off"].sum())) == expected[name]
    assert len(BASELINE_FEATURES) == 21
    assert len(PRICING_FEATURES) == 25
    assert set(PRICING_FEATURES).issubset(featured.columns)
    assert featured["credit_history_months"].notna().all()
    assert featured["fico_midpoint"].notna().all()

    print(
        f"{name}: {len(featured):,} rows | "
        f"{int(featured['charged_off'].sum()):,} charged off | "
        f"{len(BASELINE_FEATURES)} baseline features | "
        f"{len(PRICING_FEATURES)} pricing-comparison features"
    )

print("Module import and saved-data feature checks passed.")

train: 157,993 rows | 19,780 charged off | 21 baseline features | 25 pricing-comparison features
validation: 162,570 rows | 22,315 charged off | 21 baseline features | 25 pricing-comparison features
test: 283,026 rows | 42,132 charged off | 21 baseline features | 25 pricing-comparison features
Module import and saved-data feature checks passed.


rebuild preprocessors

In [19]:
from src.features.credit_features import (
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    PRICING_NUMERIC_FEATURES,
    PRICING_CATEGORICAL_FEATURES,
)

train_ready = make_row_features(splits["train"])
validation_ready = make_row_features(splits["validation"])

def build_preprocessor(numeric_columns, categorical_columns):
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                numeric_columns,
            ),
            (
                "cat",
                Pipeline([
                    (
                        "encoder",
                        OneHotEncoder(handle_unknown="ignore"),
                    ),
                ]),
                categorical_columns,
            ),
        ],
        remainder="drop",
    )

baseline_preprocessor = build_preprocessor(
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
)

pricing_preprocessor = build_preprocessor(
    NUMERIC_FEATURES + PRICING_NUMERIC_FEATURES,
    CATEGORICAL_FEATURES + PRICING_CATEGORICAL_FEATURES,
)

for name, preprocessor, columns, expected_width in [
    ("baseline", baseline_preprocessor, BASELINE_FEATURES, 98),
    ("pricing", pricing_preprocessor, PRICING_FEATURES, 142),
]:
    x_train = preprocessor.fit_transform(train_ready[columns])
    x_validation = preprocessor.transform(validation_ready[columns])

    assert x_train.shape == (157_993, expected_width)
    assert x_validation.shape == (162_570, expected_width)

    train_values = x_train.data if hasattr(x_train, "data") else np.asarray(x_train)
    val_values = (
        x_validation.data
        if hasattr(x_validation, "data")
        else np.asarray(x_validation)
    )
    assert np.isfinite(train_values).all()
    assert np.isfinite(val_values).all()

    print(
        f"{name}: train={x_train.shape}, "
        f"validation={x_validation.shape}, "
        f"all values finite"
    )

baseline: train=(157993, 98), validation=(162570, 98), all values finite
pricing: train=(157993, 142), validation=(162570, 142), all values finite


save and verify checkpoints

In [23]:
import joblib
from scipy import sparse

checkpoint_dir = project_root / "models" / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_files = {
    "baseline": checkpoint_dir / "baseline_preprocessor_2011_2013.joblib",
    "pricing": checkpoint_dir / "pricing_preprocessor_2011_2013.joblib",
}

for file in checkpoint_files.values():
    if file.exists():
        raise FileExistsError(f"Refusing to overwrite: {file}")

objects = {
    "baseline": (
        baseline_preprocessor,
        BASELINE_FEATURES,
        98,
    ),
    "pricing": (
        pricing_preprocessor,
        PRICING_FEATURES,
        142,
    ),
}

for name, (original, columns, expected_width) in objects.items():
    file = checkpoint_files[name]
    joblib.dump(original, file)
    restored = joblib.load(file)

    original_output = original.transform(validation_ready[columns])
    restored_output = restored.transform(validation_ready[columns])

    assert original_output.shape == restored_output.shape
    assert restored_output.shape == (162_570, expected_width)

    difference = original_output - restored_output
    if sparse.issparse(difference):
        assert difference.nnz == 0 or np.allclose(
            difference.data, 0, atol=1e-12
        )
    else:
        assert np.allclose(difference, 0, atol=1e-12)

    print(
        f"{name}: saved and reloaded | "
        f"{file.name} | "
        f"{file.stat().st_size / (1024**2):.2f} MiB | "
        f"validation matrix {restored_output.shape} matched"
    )

baseline: saved and reloaded | baseline_preprocessor_2011_2013.joblib | 0.01 MiB | validation matrix (162570, 98) matched
pricing: saved and reloaded | pricing_preprocessor_2011_2013.joblib | 0.01 MiB | validation matrix (162570, 142) matched


## Conclusion and handoff

The chronological splits were verified:

| Split | Issue years | Loans | Charged off | Observed rate |
|---|---|---:|---:|---:|
| Train | 2011–2013 | 157,993 | 19,780 | 12.52% |
| Validation | 2014 | 162,570 | 22,315 | 13.73% |
| Held-out test | 2015 | 283,026 | 42,132 | 14.89% |

### Deterministic row-level preparation

- Parsed `earliest_cr_line` and derived `credit_history_months` relative to each loan's `issue_month`; no dates were missing, unparseable, or later than the issue month.
- Represented missing `emp_length` explicitly as `Missing`.
- Created `annual_inc_log` and `revol_bal_log` using `log1p`. These transformations did not overwrite the processed Parquet file.
- Derived `fico_midpoint` and excluded the original FICO range endpoints from the final model input lists.
- Treated the audited `dti = 999` value as missing during feature preparation. This occurs once in the held-out test period. A diagnostic flag was created during development but was **excluded from the final model input lists**.
- Did not broadly remove observations for high income, DTI, or revolving utilisation.

The deterministic feature rules, feature lists, and chronological split definitions are saved in `src/features/credit_features.py`. The module was imported in a fresh kernel and verified against the saved PD dataset.

### Training-fitted preprocessing

A numerical median imputer and standard scaler, combined with one-hot encoding of categorical fields, were fitted on the **2011–2013 training set only**. The baseline uses 21 input features and produces 98 transformed columns. The lender-pricing comparison uses 25 input features and produces 142 transformed columns. Both produced aligned, finite training and validation matrices.

Validation contained one previously unseen `ANY` home-ownership value and one `ME` state value. `handle_unknown="ignore"` allowed these records to be transformed without fitting new categories on validation data. Neither `grade` nor `sub_grade` had unseen validation categories.

The fitted preprocessors were saved and reloaded successfully. Each reloaded object reproduced its original 2014 validation transformation:

| Artifact | Location | Output columns |
|---|---|---:|
| Baseline preprocessor | `models/checkpoints/baseline_preprocessor_2011_2013.joblib` | 98 |
| Lender-pricing preprocessor | `models/checkpoints/pricing_preprocessor_2011_2013.joblib` | 142 |

No classifier was trained in this notebook. The transformed matrices were not saved as separate files; they can be regenerated from the processed data, feature module, and fitted preprocessors.

### Saved inputs and outputs

| File | Role in the next notebooks |
|---|---|
| `data/processed/pd_cohort_2011_2015_36m.parquet` | Prepared PD cohort, including source fields, issue month, and target |
| `data/processed/loss_outcomes_2011_2015_36m.parquet` | Separate repayment-outcome table for later loss analysis; **not a PD feature source** |
| `src/features/credit_features.py` | Reusable split definitions, feature lists, and deterministic row-level rules |
| `models/checkpoints/baseline_preprocessor_2011_2013.joblib` | Training-fitted baseline preprocessing |
| `models/checkpoints/pricing_preprocessor_2011_2013.joblib` | Training-fitted lender-pricing preprocessing |

Notebook 04 can start from these saved artifacts without rerunning Notebooks 01–03. The original compressed accepted-loans file remains in `data/raw/`; it is not required for ordinary modelling from the processed cohort.

### Limitations and next step

The baseline still requires a feature-timing audit, particularly for `verification_status`. The grade/pricing comparison includes Lending Club's own underwriting outputs and therefore answers a different question from an independent pre-decision model. The target represents observed final `Charged Off` versus `Fully Paid` status, **not** a fixed-horizon default probability.

The saved `.joblib` files contain fitted preprocessing, **not** classifiers or calibrated PD models. They should be loaded only from this trusted project and used with a compatible software environment. No LGD, EAD, fairness audit, or Expected Loss model has been completed here.

**Next:** Notebook 04 loads the saved PD cohort, imports `credit_features.py`, and loads the two fitted preprocessors. It compares model benchmarks and makes calibration and threshold decisions using development data, then evaluates the selected approach on the held-out 2015 test period.